### Purpose

품목·거래처 **등록 요청** 1단계 배포 (docs/gdb-request-registration-queue.md 4장, 10장 G1~G3).

| 대상 | 내용 |
|---|---|
| Lambda `dsglobaldbCreate` (갱신) | create 에 `regStatus` `ecountStatus` `regRequest` `regEvidence` `regDecision`, 증빙 자동 등록 판단 |
| 환경 변수 `KEY_APPS` | `{sha256(키 값): "inv"/"quote"/"web"}` — 요청한 앱을 키로 정한다 (키 값은 넣지 않음) |
| 새 경로 (API 키 필요) | `POST /dscustomer/evidence-upload`, `GET /regrequests`, `POST /dschemical/review`, `POST /dscustomer/review` |
| S3 `dsbaseinfo` | `evidence/` 수명주기 5년, 버킷 CORS (브라우저에서 presigned PUT·GET) |
| IAM `DSmapLambda` | `dsbaseinfo/evidence/*` 읽기·쓰기 (인라인 정책 `DSGlobalDBEvidence`) |

- review 는 게이트웨이에서 키를 검사하고, Lambda 가 한 번 더 **`globaldb-inv` 키만** 받는다 (다른 키 403)
- **create 의 키 필수는 아직 켜지 않는다** (4.7 배포 순서 4단계). 키 없는 create 는 `regRequest.app = "unknown"` 으로 기록
- CORS 는 버킷 전체에 걸리지만 서명된 주소(presigned)로만 읽고 쓸 수 있으므로 공개되는 것은 없다
- 배포 전 테스트: `pytest AWS/Lambda/dsglobaldbCreate`
- **마지막 셀(`dev` 스테이지 배포)은 API 전체를 다시 배포한다**

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import hashlib
import json
import urllib.error
import urllib.request

import boto3

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
from deploy_helpers import (deploy_lambda, ensure_resource_path, get_request_template, setup_lambda_method,
                            setup_options, grant_invoke, deploy_stage_and_check)

Settings

In [ ]:
ROLE_NAME = 'DSmapLambda'
ROLE_ARN = 'arn:aws:iam::199525800117:role/DSmapLambda'
API_ID = 'jyipsj28s9'
STAGE = 'dev'
TEMPLATE_FROM = '/dschemical'
FUNCTION_NAME = 'dsglobaldbCreate'
SOURCE_FILE = './dsglobaldbCreate/lambda_function.py'

# 경로 → [(메서드, API 키 필요)]
ROUTES = {
    'dscustomer/evidence-upload': [('POST', True)],
    'regrequests': [('GET', True)],
    'dschemical/review': [('POST', True)],
    'dscustomer/review': [('POST', True)],
}
KEY_APPS = {'globaldb-inv': 'inv', 'globaldb-quote': 'quote', 'globaldb-web': 'web'}   # API 키 이름 → 앱

BUCKET = 'dsbaseinfo'
EVIDENCE_PREFIX = 'evidence/'
EVIDENCE_DAYS = 1827            # 5년 (세금계산서 보관 기간)
LIFECYCLE_RULE_ID = 'evidence-5y'
CORS_RULE_ID = 'globaldb-evidence'
POLICY_NAME = 'DSGlobalDBEvidence'
ACCOUNT_ID = ROLE_ARN.split(':')[4]
BASE_URL = f'https://{API_ID}.execute-api.us-east-1.amazonaws.com/{STAGE}'

1. IAM — Lambda 역할에 증빙 경로 읽기·쓰기 (presigned 주소는 Lambda 역할 권한으로 서명된다)

In [ ]:
iam = boto3.client('iam')
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName=POLICY_NAME, PolicyDocument=json.dumps({
    'Version': '2012-10-17',
    'Statement': [{'Effect': 'Allow', 'Action': ['s3:PutObject', 's3:GetObject'],
                   'Resource': f'arn:aws:s3:::{BUCKET}/{EVIDENCE_PREFIX}*'}],
}))
print('인라인 정책:', ROLE_NAME, POLICY_NAME)

2. S3 — `evidence/` 수명주기 5년 + 버킷 CORS (있던 규칙은 그대로 두고 이 규칙만 넣거나 바꾼다)

In [ ]:
s3 = boto3.client('s3')

try:
    rules = s3.get_bucket_lifecycle_configuration(Bucket=BUCKET)['Rules']
except s3.exceptions.ClientError as e:
    if e.response['Error']['Code'] != 'NoSuchLifecycleConfiguration':
        raise
    rules = []
rules = [r for r in rules if r.get('ID') != LIFECYCLE_RULE_ID] + [{
    'ID': LIFECYCLE_RULE_ID, 'Status': 'Enabled', 'Filter': {'Prefix': EVIDENCE_PREFIX},
    'Expiration': {'Days': EVIDENCE_DAYS},
}]
s3.put_bucket_lifecycle_configuration(Bucket=BUCKET, LifecycleConfiguration={'Rules': rules})
print('수명주기 규칙:', [r['ID'] for r in rules])

try:
    cors = s3.get_bucket_cors(Bucket=BUCKET)['CORSRules']
except s3.exceptions.ClientError as e:
    if e.response['Error']['Code'] != 'NoSuchCORSConfiguration':
        raise
    cors = []
cors = [r for r in cors if r.get('ID') != CORS_RULE_ID] + [{
    'ID': CORS_RULE_ID, 'AllowedMethods': ['PUT', 'GET', 'HEAD'], 'AllowedOrigins': ['*'],
    'AllowedHeaders': ['*'], 'MaxAgeSeconds': 3000,
}]
s3.put_bucket_cors(Bucket=BUCKET, CORSConfiguration={'CORSRules': cors})
print('CORS 규칙:', [r.get('ID') for r in cors])

3. Lambda 갱신 + `KEY_APPS` 환경 변수 (키 값의 해시만, 값은 출력하지 않음)

In [ ]:
function_arn = deploy_lambda(FUNCTION_NAME, SOURCE_FILE, ROLE_ARN,
                             description='글로벌 DB 안전한 쓰기 (create · update/alias/history · 등록 요청 review)',
                             timeout=29, memory=256)

apigw = boto3.client('apigateway')
keys = {k['name']: k['value'] for k in apigw.get_api_keys(includeValues=True, limit=500)['items'] if k['name'] in KEY_APPS}
assert set(keys) == set(KEY_APPS), f'API 키가 없습니다: {set(KEY_APPS) - set(keys)}'
key_apps = {hashlib.sha256(value.encode()).hexdigest(): KEY_APPS[name] for name, value in keys.items()}

lam = boto3.client('lambda')
env = lam.get_function_configuration(FunctionName=FUNCTION_NAME).get('Environment', {}).get('Variables', {})
env['KEY_APPS'] = json.dumps(key_apps)
lam.update_function_configuration(FunctionName=FUNCTION_NAME, Environment={'Variables': env})
lam.get_waiter('function_updated_v2').wait(FunctionName=FUNCTION_NAME)
print('KEY_APPS 설정:', sorted(key_apps.values()))

4. API Gateway 경로 (모두 API 키 필요) + 호출 권한

In [ ]:
request_template = get_request_template(API_ID, TEMPLATE_FROM)
for path, methods in ROUTES.items():
    resource_id = ensure_resource_path(API_ID, path)
    for method, key_required in methods:
        setup_lambda_method(API_ID, resource_id, method, function_arn, request_template, api_key_required=key_required)
    setup_options(API_ID, resource_id, [m for m, _ in methods])
    grant_invoke(FUNCTION_NAME, API_ID, ACCOUNT_ID, path)

5. `dev` 스테이지 배포 + 확인 (데이터를 쓰지 않음. 증빙 업로드 확인 파일은 바로 지운다)

In [ ]:
deploy_stage_and_check(API_ID, STAGE, 'dschemical/history?id=__check__',
                       'registration queue: evidence-upload, regrequests, review')

import time

def call(method, path, body=None, key=None):
    # 새 경로는 모든 서버에 퍼질 때까지 1~2분 'Missing Authentication Token'(403)이 섞여 나온다 → 키가 있으면 다시 시도
    headers = {'Content-Type': 'application/json', **({'x-api-key': keys[key]} if key else {})}
    for _ in range(15):
        req = urllib.request.Request(BASE_URL + path, method=method, headers=headers,
                                     data=json.dumps(body).encode() if body is not None else None)
        try:
            with urllib.request.urlopen(req) as r:
                res = json.load(r)
                return res['statusCode'], json.loads(res['body'])
        except urllib.error.HTTPError as e:
            if not (key and b'Missing Authentication Token' in e.read()):
                return e.code, None
            time.sleep(8)
    raise RuntimeError(f'{path}: 2분 넘게 경로가 반영되지 않았습니다')

status, _ = call('GET', '/regrequests')
print('키 없이 regrequests:', status)                              # 403 (게이트웨이)
status, rows = call('GET', '/regrequests?pending=any', key='globaldb-web')
print('웹 키로 regrequests:', status, f'{len(rows)}건' if status == 200 else rows)
status, body = call('POST', '/dschemical/review', {'id': '__check__', 'decision': 'approve', 'by': 'check'}, key='globaldb-quote')
print('견적 키로 review:', status, body)                           # 403 (Lambda)
status, body = call('POST', '/dschemical/review', {'id': '__check__', 'decision': 'approve', 'by': 'check'}, key='globaldb-inv')
print('재고 키로 없는 코드 review:', status, body)                  # 404

status, upload = call('POST', '/dscustomer/evidence-upload', {'docType': '영수증', 'contentType': 'image/png'}, key='globaldb-web')
print('증빙 업로드 주소:', status, upload['fileKey'])
put = urllib.request.Request(upload['uploadUrl'], method='PUT', data=b'check', headers={'Content-Type': 'image/png'})
with urllib.request.urlopen(put) as r:
    print('presigned PUT:', r.status)
object_key = 'evidence/customers/' + upload['fileKey']
print('올라감:', s3.head_object(Bucket=BUCKET, Key=object_key)['ContentLength'], 'bytes')
s3.delete_object(Bucket=BUCKET, Key=object_key)
print('확인 파일 지움')